除了 ZeRO 和 Offload 这俩显存优化的核心，DeepSpeed 还有几个在实际落地中非常重要、高频使用的模块。它们分别瞄准 通信瓶颈、计算效率、推理部署 和 完整训练流水线。


---

## 符号约定

- **模型参数**：$\theta = \{W_1, W_2, \dots, W_L\}$，总参数量 $\Psi = \sum_{l=1}^{L} |W_l|$（元素个数）。
- **数据并行度**：$N$，GPU 索引 $k \in \{0,\dots,N-1\}$。
- **批次**：全局批次 $\mathcal{B}$，切分为局部批次 $\mathcal{B}_k$，$|\mathcal{B}_k| = B/N$。
- **损失**：$\mathcal{L}(\theta) = \frac{1}{N}\sum_{k=0}^{N-1} L_k(\theta)$，其中 $L_k(\theta) = \frac{1}{|\mathcal{B}_k|}\sum_{i\in\mathcal{B}_k} \ell(f(x^{(i)};\theta), y^{(i)})$。
- **优化器**：Adam，学习率 $\eta$，$\beta_1,\beta_2$，$\epsilon$。
- **精度**：FP16（2字节），FP32（4字节）。模型状态总显存 $16\Psi$（参数 $2\Psi$，梯度 $2\Psi$，优化器状态 $12\Psi$）。

---

## 一、通信优化：打破带宽墙

### 1.1 1-bit Adam / 0/1 Adam
**核心原理**：在分布式 All-Reduce 之前，将 FP32/FP16 梯度压缩为 **1 比特**（仅保留符号），接收端解压为 $\pm 1$。为弥补压缩带来的信息损失，引入**误差反馈**。

设第 $t$ 步 GPU $k$ 的原始局部梯度为 $g_k^{(t)}$（FP32），压缩操作 $\mathcal{C}$ 为：
$$
\mathcal{C}(g_k^{(t)}) = \|g_k^{(t)}\|_1 \cdot \text{sign}(g_k^{(t)}) / d
$$
其中 $d$ 为梯度张量元素数，使得解压后是单位长度的 $\pm 1$ 向量。

**误差反馈**：维护误差累积 $\delta_k^{(t)}$：
$$
\delta_k^{(t+1)} = g_k^{(t)} + \delta_k^{(t)} - \text{decompress}(\mathcal{C}(g_k^{(t)} + \delta_k^{(t)}))
$$
通信前对 $g_k^{(t)} + \delta_k^{(t)}$ 做压缩，保证长期无偏。

**0/1 Adam 变体**：对零梯度直接用零值编码，进一步稀疏通信量。

压缩比为 $32:1$（FP32→1 bit），加上误差反馈，通信量降至原来的 $1/5 \sim 1/20$，收敛几乎无影响。

### 1.2 DeepSpeed-Ulysses（序列并行）
**目标**：训练超长序列（如 128K token），注意力矩阵 $QK^\top$ 与序列长度 $S$ 成 $O(S^2)$ 关系，显存炸裂。

**方法**：将序列维度切分到 $N$ 张 GPU，通过 **All-to-All** 通信重排数据。

设每 GPU 的输入张量形状为 $(B, S/N, d_{model})$，有 $h$ 个注意力头。

- **All-to-All 前向**：
  - 将每个 GPU 上的 $(B, S/N, h, d_h)$ 重排为 $(B, h, S/N, d_h)$，然后沿着 $h$ 维度切分并做 All-to-All。
  - 通信后，GPU $i$ 获得所有序列位置但仅负责 $h/N$ 个注意力头的数据。
  - 执行局部注意力（每个 GPU 算完整的序列的 $h/N$ 个头）。
  - 再通过反向 All-to-All 将结果拼接回各 GPU。

**通信量**：All-to-All 的通信量约 $O(B \times S \times d_{model})$，与序列长度呈线性，而与 $S^2$ 注意力计算解耦。适合超长上下文训练。

---

## 二、3D 并行：张量、流水线与数据的融合

### 2.1 张量并行（Tensor Parallelism, TP）
**层内切分**：将单个 Transformer 层的权重矩阵按行或列切分。

**列切分（Column Parallel）**：
线性层 $Y = XW$，将 $W \in \mathbb{R}^{d_{in} \times d_{out}}$ 按列切为 $[W_1, W_2]$ 分给两张 GPU。输入 $X$ 保持不变。输出：
$$
Y_1 = XW_1, \quad Y_2 = XW_2
$$
随后 **All-Gather** 得到 $Y = [Y_1, Y_2]$。前向通信量：每列切分层 All-Gather 输出，约 $2\Psi_{\text{layer}}$（$\Psi_{\text{layer}}$ 为该层输出元素数）。

**行切分（Row Parallel）**：
将 $W$ 按行切分，同时 $X$ 按列切分为 $[X_1, X_2]$。输出 $Y = X_1W_1 + X_2W_2$。前向需要 **All-Reduce** 求和。通信量类似。

**典型 Transformer 层的 TP 设计**：
- Attention QKV 投影：列切分。
- Attention 输出投影：行切分。
- FFN 第一层：列切分（加 GELU）。
- FFN 第二层：行切分。

这样每一层前向需要一次 All-Reduce（行切分输出）和若干次 All-Gather，后向对称。通信量虽大，但限制在 NVLink 域内。

### 2.2 流水线并行（Pipeline Parallelism, PP）
**层间切分**：将 $L$ 层划分为 $P$ 个 stage，每个 stage 分配到一个 GPU。

**1F1B 调度**（One Forward, One Backward）：将 batch 切成 $M$ 个 micro-batch。流水线开始后，每个 GPU 交替执行前向和反向：
- 前向预热阶段：连续注入 micro-batch 直到最深 stage。
- 稳定阶段：每完成一个 micro-batch 的前向，立即执行一个 micro-batch 的反向。
气泡时间占比为：
$$
\text{Bubble} = \frac{P-1}{M+P-1}
$$
通过增大 $M$，气泡可趋于 $0$。

### 2.3 三维融合与 ZeRO 的关系
- 先分配 **PP 维度**，将模型层按组划分到不同机器。
- 每组内使用 **TP** 切分单层。
- 若有多余机器做相同配置的副本，则构成 **DP 维度**，并在 DP 组内启用 **ZeRO-1 或 ZeRO-2**，分片优化器状态和梯度。
- 一般不启用 ZeRO-3，因为 TP 已经切了参数，ZeRO-3 的额外通信会得不偿失。

---

## 三、分区激活检查点（Partitioned Activation Checkpointing）

### 3.1 传统激活检查点
前向仅保留各层的**输入激活** $a^{(l-1)}$ 作为检查点，丢弃中间 $z^{(l)}, a^{(l)}$。反向时：
$$
z^{(l)} = a^{(l-1)} W_l, \quad a^{(l)} = \sigma(z^{(l)}) \quad \text{(重算)}
$$
然后计算 $\delta^{(l-1)}$ 和 $\nabla W_l$。

单张 GPU 保留的激活值大小为 $O(S \times d_{model})$，乘以层数。长序列下仍然很大。

### 3.2 分区激活检查点
**观察**：数据并行下，所有 GPU 使用相同模型参数，输入数据是按 batch 切分的不同部分，但检查点激活 $a^{(l-1)}$ 在相同模型、相同输入（因为是同一 batch 的不同分片）下，是按 batch 维度不同，而非冗余。纠正：数据并行下，各 GPU 的输入数据不同，因此计算出的 $a^{(l-1)}$ **并不相同**，不能通过分片消除冗余。分区激活检查点的真实原理是沿 **序列长度** 或 **batch** 维度切分**保存的检查点**，通过分散存储来降低单卡内存峰值。

假设采用 ZeRO-3 的**分区激活检查点**（DeepSpeed 实现）：
- 前向计算完毕后，将每个检查点 $a^{(l-1)}$ 沿 batch 维度切分为 $N$ 片，每张卡保留 $1/N$。
- 反向需要时，**All-Gather** 拼回完整 $a^{(l-1)}$ 以便重算。
- 重算后立即释放拼回的激活值。

**存储节省**：单卡激活值留存从原本的 $O(L \times B \times S \times d_{model})$ 降为 $1/N$。

**额外通信**：每层反向时一次 All-Gather（大小等同于一层激活值），可与参数 All-Gather 和梯度 Reduce-Scatter 重叠。

---

## 四、ZeRO-MoE 与专家并行

### 4.1 MoE 层结构
一个 MoE 层由门控网络 $G(x)$ 和 $E$ 个专家 $\{E_1, \dots, E_E\}$ 组成。对于输入 token $x$：
- 门控输出概率分布 $p = \text{softmax}(x W_g)$。
- 选择 $K$ 个最高概率的专家（Top-K），得到权重 $g_i$。
- 输出：
$$
y = \sum_{i \in \text{Top-K}} g_i \cdot E_i(x)
$$

### 4.2 专家并行（Expert Parallelism）
**分布**：将 $E$ 个专家分配到 $N$ 张 GPU，每卡持有 $E/N$ 个专家的参数。

**All-to-All 通信**：
- 各 GPU 计算门控后，确定每个 token 的目标专家位置。
- 执行 **All-to-All**：将 token 的隐状态向量发送到持有目标专家的 GPU。
- 目标 GPU 计算本地专家的输出。
- 再次 **All-to-All**：将输出传回原 GPU 进行后续计算。

通信量：每层 MoE 两次 All-to-All，总数据量约 $4 \times B \times S \times d_{model}$（前向+反向加倍）。

**优化**：DeepSpeed 将连续小 All-to-All 合并为大块传输，并根据拓扑调度。

### 4.3 负载均衡损失
为防止专家使用不均，加入辅助损失：
$$
L_{\text{aux}} = \alpha \cdot E \cdot \sum_{i=1}^{E} f_i \cdot P_i
$$
其中 $f_i$ 是路由到专家 $i$ 的 token 比例（期望均匀 $1/E$），$P_i$ 是门控分配给专家 $i$ 的平均概率。最小化此损失鼓励均匀路由。

### 4.4 与 ZeRO 的结合
- **非专家参数**（Attention、门控等）：使用 ZeRO-1/2 分片优化器状态和梯度。
- **专家参数**：通过专家并行分布在 GPU 上，无需 ZeRO 分片（本身已是唯一存储）。

---

## 五、计算优化：内核融合与推理加速

### 5.1 内核融合（Kernel Fusion）
**原理**：将多个内存受限的细碎 CUDA kernel 合并为一个大 kernel，消除中间结果的显存读写。

例如，将 `y = x + dropout(gelu(layernorm(x)W + b))` 中的 Layernorm、GEMM、激活、Dropout、残差加法等融合成单个 kernel。

**性能公式**：若未融合，每步都要读写 HBM，总时间由带宽限制：
$$
T_{\text{mem}} = \frac{D_{\text{read}} + D_{\text{write}}}{B_{\text{HBM}}}
$$
融合后，$D_{\text{read/write}}$ 大幅减少，$T_{\text{mem}}$ 降低，使得实际计算时间更接近理论算力峰值 $T_{\text{comp}} = \frac{\text{FLOPs}}{\text{TFLOPS}}$。

DeepSpeed 的 Fused Transformer Kernel 能将 Attention 和 FFN 的访存压缩 60% 以上。

### 5.2 DeepSpeed-Inference / FastGen
**推理引擎核心**：
- 集成融合 kernel。
- 自动张量并行：给定 `tensor_parallel_size`，自动切分 HF 模型。
- 动态批处理：合并不同请求的 tokens，提高 GPU 利用率。

**FastGen 调度**：
将 LLM 生成分离为**预填充**（并行处理 prompt）和**解码**（逐 token 生成）。通过连续批处理动态分配计算资源，吞吐量提升可达 2×。

### 5.3 ZeRO-Inference
**原理**：将训练好的模型权重沿 DP 维度切分，每张 GPU 仅持有 $1/N$ 权重分片，其余存储在 CPU 内存或 NVMe。推理时逐层 **All-Gather** 拼出完整权重，计算后释放。

总显存需求从 $2\Psi$ 降至 $2\Psi/N + \text{工作缓冲区}$，使得单张 24GB GPU 可推理 176B 模型。

---

## 六、完整训练流水线与辅助工具

### 6.1 DeepSpeed-Chat（RLHF）
**三阶段流程**：
1. **SFT**：监督微调基座模型，使用 ZeRO。
2. **Reward Model**：训练奖励模型，同样用 ZeRO。
3. **PPO 强化学习**：
   - 同时运行训练引擎（Actor，开启 ZeRO-3/Offload）和推理引擎（Frozen Reference Model，开启 TP/融合 kernel）。
   - Hybrid Engine 动态切换训练/推理模式，重用显存。

PPO 目标函数（简化）：
$$
L^{\text{PPO}}(\theta) = \mathbb{E}\left[ \min\left( r_t(\theta) \hat{A}_t, \ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right] - \beta \cdot \text{KL}(\pi_{\theta} \| \pi_{\text{ref}})
$$
其中 $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$，$\hat{A}_t$ 为优势函数。

### 6.2 多模态支持
类似 RLHF，将训练流水线扩展到视觉-语言模型（如 VisualChat），使用 ZeRO 和 TP 联合优化。

### 6.3 Autotuning（自动调优）
目标函数：在给定显存约束下最大化吞吐量（样本/秒）。自动搜索变量包括：
- `train_batch_size`
- `gradient_accumulation_steps`
- ZeRO 阶段（1/2/3）
- 卸载开关
- 通信桶大小等

通过在小型测试运行中测量显存和速度，建立成本模型，选择 Pareto 最优配置。

### 6.4 数据效率与课程学习
**课程学习**：训练时序列长度 $S$ 随时间增加：
$$
S(t) = S_{\min} + (S_{\max} - S_{\min}) \cdot \min\left(1, \frac{t}{T_{\text{warmup}}}\right)
$$
在初期使用短序列，大量减少计算量，加速预训练。

### 6.5 模型压缩
- **量化**：将 FP16 权重/激活量化为 INT8/INT4。对每个张量计算 scale $s$ 和零点 $z$：
  $$
  x_{\text{quant}} = \text{round}(x / s) + z
  $$
- **剪枝**：移除接近零的权重。
- **蒸馏**：用大模型指导小模型训练。

---

## 七、技术全景总结

| 场景需求 | 核心技术组合 | 关键公式/原理 | 核心收益 |
|----------|-------------|--------------|----------|
| **单卡/少卡显存不足** | ZeRO-3 + Offload/Infinity | $16\Psi \to 16\Psi/N$，三级卸载 | 单卡跑百亿模型，多卡万亿 |
| **跨机通信瓶颈** | 1-bit Adam / 0/1 Adam | 梯度压缩 $32\times$，误差反馈 | 通信量降低 5-20 倍 |
| **超长序列训练** | Ulysses + 分区激活检查点 | All-to-All 重排，激活分片存储 | 支持百万 token 序列 |
| **单层权重放不下** | 3D 并行 (TP + PP + ZeRO) | TP 列/行切分，1F1B 调度，气泡公式 | 支撑千亿稠密模型 |
| **万亿参数稀疏模型** | ZeRO-MoE + 专家并行 | Top-K 门控，All-to-All，负载均衡损失 | 容量剧增，计算不涨 |
| **高吞吐推理** | DeepSpeed-Inference + 内核融合 | 融合 kernel，访存减少 60%+，FastGen 调度 | 低延迟，单卡跑大模型 |
| **RLHF 训练** | DeepSpeed-Chat + Hybrid Engine | PPO 目标函数，训练/推理模式切换 | 一体化流程，显存时间双省 |
| **自动配置** | Autotuning | 成本模型，Pareto 搜索 | 免手动调参 |

所有这些技术的内核一致：**通过分片、卸载、融合、压缩等手段，将存储和通信的开销降到最低，将算力利用率推到极限**。这正是 DeepSpeed 从显存优化引擎演变为完整大模型基础设施的核心逻辑。